IN Phase 1 I assume we have 
1. Fetched all the transcripts from YT
2. Split them 
3. Saved them in our chroma Vector DB (Ekantik Project/vector_DB)
4. dot_env

In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
os.environ[ 'HF_HOME'] = '/Users/tejas/Documents/LangChain/Ekantik Project/embedding_model'
embedding_model = HuggingFaceEmbeddings(
   model_name="sentence-transformers/LaBSE"
)

# dot_env
from dotenv import load_dotenv
import os

# load .env from 2 directories up
load_dotenv("../../.env")
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function= embedding_model,
    persist_directory = "/Users/tejas/Documents/LangChain/Ekantik Project/vector_DB",# location I will store vectors
    collection_name="Ekantik_Vartalap", # name of the db / collection
)


/var/folders/q3/fl0ndk4j7v39n4nsq38qdk4w0000gn/T/ipykernel_43269/3911522662.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/opt/anaconda3/envs/pyTensor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/q3/fl0ndk4j7v39n4nsq38qdk4w0000gn/T/ipykernel_43269/3911522662.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chr

From here we will create a retriever 

In [2]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7,"lambda_mult":1})

mmr_result=retriever.invoke("यह देहराग कैसे छूटे ?")
print(mmr_result[3].page_content)

मेरा उद्धार कैसे होगा महाराज जी? नाम
संकीर्तनम यस सर्व पाप प्रणाशनम
प्रणामो दुख शमनम तम नमामि हरि परम भगवान
को दंडवत करो 108 दंडवत रोज नियम से भगवान
की करो शरीर स्वस्थ है और खूब नाम कीर्तन
करो नाम जप करो आप क्या
दूसरों को पवित्र करने वाले बन जाएंगे आप
स्वयं पवित्र हो जाएंगे सतरति सतरति
सलोकाम तारति मम भक्ति युक्त भन पुना अरे
डरो मत आज से पाप आपका नियम लो कि मैं पाप
नहीं करूंगा। अब नसान यम ना नसो और डट के
नाम कीर्तन भगवान को दंडवत नाम जप भगवान
की लीला कथा श्रवण सबका भस्म हो जाएगा।
उसी कारण से हमको लगता है शंकर भगवान छोड़
दिए भगवान भगवान नहीं छोड़ते भगवान नहीं
छोड़ते अब राम नाम बहुत खूब खूब नाम जप
करो भगवान का स्वभाव समझ लो वो कभी किसी
को छोड़ते नहीं उनका स्वभाव नहीं हमारे
भगवान का विमुख भय निमिशो ना अकपा जब चित
तब वैसे ऐसा करुणामय स्वभाव है विमुख भय
निमिशो ना अकृपा जब चित तब वैसे परम
करुणामय भगवान है पर उनकी करुणा की तो
पूछो मत हमारी मलिनता है जो उनकी करुणा की
तरफ नहीं देख पा रही है। तुम नाम कीर्तन
करो, नाम जप करो इसी से पवित्र हो जाओगे।
यात्रा कर रहे हैं तो हम साधु संत का संग
किए हैं औ

Augmentation 

In [3]:

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["docs", "query"],
    validate_template=True,
    template="""
आप श्री प्रेमानंद जी महाराज के प्रवचनों पर आधारित उत्तर देने वाले सहायक हैं।

नीचे दिए गए प्रवचन अंश "एकांतिक वार्तालाप" से लिए गए हैं।
आपको **केवल इन्हीं अंशों के आधार पर** उत्तर देना है।

--------------------
प्रवचन अंश:
{docs}
--------------------

प्रश्न:
{query}

निर्देश:
- उत्तर 6–10 पंक्तियों में स्पष्ट रूप से दें
- उत्तर अधूरा न छोड़ा जाए
- उत्तर केवल दिए गए प्रवचन अंशों पर आधारित हो
- अपनी ओर से कोई नई बात न जोड़ें
- यदि एक से अधिक एकांतिक का संदर्भ हो, तो सभी का उल्लेख करें
- उत्तर के अंत में संबंधित declared_ekantik_number अवश्य लिखें
- यदि उत्तर स्पष्ट रूप से उपलब्ध न हो, तो साफ लिखें:
  "इस प्रश्न का उत्तर दिए गए प्रवचनों में स्पष्ट रूप से नहीं मिलता"

उत्तर:
"""
)

query = "परमात्मा हमें गलत कर्म करने से क्यों नहीं रोकते?"
docs = retriever.invoke(query)
final_doc=""
for i in docs:

    final_doc+= (

      "\n==========\n"
      f"एकांतिक क्रमांक: {i.metadata['declared_ekantik_number']}\n"
      f"वीडियो ID: {i.metadata['video_id']}\n"
      "\n==========\n"
      f"{i.page_content}\n"
    )

In [4]:
from langchain_groq import ChatGroq

language_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,

)
final_response = language_model.invoke(prompt.invoke({"docs":final_doc,"query":query}))

In [5]:
print(final_response)

content='परमात्मा हमारे मन‑इंद्रियों को स्वतंत्रता देता है, इसलिए वह हमारे हर कदम को रोक नहीं पाते; जब हम बुरे विचारों और इंद्रियों के कारण गलत कर्म करते हैं तो वह हमें निराश नहीं करते, बल्कि पुनः प्रयास करने, नाम‑जप और सत्संग से सुधरने का अवसर देते हैं।  \nहमारी त्रुटि इस बात में है कि हम भगवान द्वारा स्थापित नियमों के अनुसार नहीं चलते, इसलिए बुरे कर्मों का फल हमें स्वयं भुगतना पड़ता है।  \nभगवान की कृपा यह है कि वह हमारे द्वारा किए गये पाप को नष्ट करने के लिये नाम‑जप को साधन बनाते हैं, परन्तु वह हमारे चयन को रोक नहीं सकते।  \nइस प्रकार, गलत कर्म करने से रोकना हमारे अपने मन‑बुद्धि और आत्म‑नियंत्रण पर निर्भर है, जबकि परमात्मा केवल मार्गदर्शन और क्षमा प्रदान करते हैं।  \n\nसंदर्भ: 949, 816, 446, 1026  \ndeclared_ekantik_number: 949, 816, 446, 1026' additional_kwargs={'reasoning_content': 'We need to answer: "परमात्मा हमें गलत कर्म करने से क्यों नहीं रोकते?" Using only given excerpts. Need 6-10 lines, mention all relevant ekantik numbers, and at end include declared_ekantik_number (maybe

In [6]:
print(final_response.content)


परमात्मा हमारे मन‑इंद्रियों को स्वतंत्रता देता है, इसलिए वह हमारे हर कदम को रोक नहीं पाते; जब हम बुरे विचारों और इंद्रियों के कारण गलत कर्म करते हैं तो वह हमें निराश नहीं करते, बल्कि पुनः प्रयास करने, नाम‑जप और सत्संग से सुधरने का अवसर देते हैं।  
हमारी त्रुटि इस बात में है कि हम भगवान द्वारा स्थापित नियमों के अनुसार नहीं चलते, इसलिए बुरे कर्मों का फल हमें स्वयं भुगतना पड़ता है।  
भगवान की कृपा यह है कि वह हमारे द्वारा किए गये पाप को नष्ट करने के लिये नाम‑जप को साधन बनाते हैं, परन्तु वह हमारे चयन को रोक नहीं सकते।  
इस प्रकार, गलत कर्म करने से रोकना हमारे अपने मन‑बुद्धि और आत्म‑नियंत्रण पर निर्भर है, जबकि परमात्मा केवल मार्गदर्शन और क्षमा प्रदान करते हैं।  

संदर्भ: 949, 816, 446, 1026  
declared_ekantik_number: 949, 816, 446, 1026
